In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_1.py ──
"""
Shared infrastructure for MLFP02 Exercise 1 — Probability and Bayesian
Fundamentals.

Contains: HDB data loading (4-ROOM 2020+ slice), prior/posterior math for
the Normal-Normal and Beta-Binomial conjugate families, bootstrap helpers,
and output directory wiring.

Technique-specific narrative (MLE derivation, Bayes scenarios, credible vs
confidence simulation, bootstrap comparison) does NOT belong here — it
lives in the per-technique files.
"""

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import polars as pl
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

# Output directory for plots (HTML) — every technique writes here.
OUTPUT_DIR = Path("outputs") / "mlfp02_ex1_bayes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Canonical prior used across all four technique files. A single source
# of truth means Task 5's prior sweep and Task 4's posterior use the same
# starting point.
PRIOR_MU_0: float = 500_000.0  # SGD — Singapore 4-room HDB market anchor
PRIOR_SIGMA_0: float = 100_000.0  # SGD — moderate uncertainty

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale flats (Singapore, data.gov.sg)
# ════════════════════════════════════════════════════════════════════════

_loader = MLFPDataLoader()


def load_hdb_all() -> pl.DataFrame:
    """Full HDB resale dataset filtered to 2020+ transactions.

    Returns a polars DataFrame with columns: month, town, flat_type,
    resale_price, plus any others present in the source parquet. Used
    by Task 1 (truth tables) and Task 8 (expected value by flat type).
    """
    hdb = _loader.load("mlfp01", "hdb_resale.parquet")
    return hdb.filter(pl.col("month").str.to_date("%Y-%m") >= pl.date(2020, 1, 1))


def load_hdb_4room() -> pl.DataFrame:
    """4-ROOM slice of HDB resale (2020+) — the primary estimation target."""
    return load_hdb_all().filter(pl.col("flat_type") == "4 ROOM")


def load_hdb_prices_4room() -> np.ndarray:
    """Return the 4-room resale_price column as a float64 numpy array.

    This is the primary observation vector for MLE, Normal-Normal, and
    bootstrap tasks.
    """
    return load_hdb_4room()["resale_price"].to_numpy().astype(np.float64)


# ════════════════════════════════════════════════════════════════════════
# NORMAL-NORMAL CONJUGATE
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class NormalPosterior:
    """Posterior for μ under a Normal-Normal conjugate with known σ."""

    mean: float
    std: float
    prior_mean: float
    prior_std: float
    n: int
    sigma_known: float

    @property
    def precision_prior(self) -> float:
        return 1.0 / self.prior_std**2

    @property
    def precision_data(self) -> float:
        return self.n / self.sigma_known**2

    @property
    def prior_weight(self) -> float:
        """Fraction of posterior precision contributed by the prior (0..1)."""
        return self.precision_prior / (self.precision_prior + self.precision_data)

    def credible_interval(self, level: float = 0.95) -> tuple[float, float]:
        z = stats.norm.ppf(0.5 + level / 2)
        return (self.mean - z * self.std, self.mean + z * self.std)


def normal_normal_posterior(
    data: np.ndarray,
    prior_mean: float,
    prior_std: float,
    sigma_known: float,
) -> NormalPosterior:
    """Closed-form posterior for μ under N(μ₀, σ₀²) prior and known σ.

    Posterior precision = prior precision + n / σ². Posterior mean is the
    precision-weighted average of the prior mean and the sample mean.
    """
    n = len(data)
    xbar = float(np.mean(data))
    prec_prior = 1.0 / prior_std**2
    prec_data = n / sigma_known**2
    prec_post = prec_prior + prec_data
    sigma_post_sq = 1.0 / prec_post
    mu_post = sigma_post_sq * (prior_mean * prec_prior + n * xbar / sigma_known**2)
    return NormalPosterior(
        mean=mu_post,
        std=float(np.sqrt(sigma_post_sq)),
        prior_mean=prior_mean,
        prior_std=prior_std,
        n=n,
        sigma_known=sigma_known,
    )


# ════════════════════════════════════════════════════════════════════════
# BETA-BINOMIAL CONJUGATE
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class BetaPosterior:
    """Posterior for p under a Beta-Binomial conjugate."""

    alpha: float
    beta: float
    prior_alpha: float
    prior_beta: float
    k: int
    n: int

    @property
    def mean(self) -> float:
        return self.alpha / (self.alpha + self.beta)

    @property
    def prior_mean(self) -> float:
        return self.prior_alpha / (self.prior_alpha + self.prior_beta)

    def credible_interval(self, level: float = 0.95) -> tuple[float, float]:
        lo = (1 - level) / 2
        hi = 1 - lo
        return tuple(stats.beta.ppf([lo, hi], self.alpha, self.beta).tolist())


def beta_binomial_posterior(
    k: int, n: int, prior_alpha: float, prior_beta: float
) -> BetaPosterior:
    """Closed-form posterior for p under Beta(α, β) prior and k/n Binomial."""
    return BetaPosterior(
        alpha=prior_alpha + k,
        beta=prior_beta + (n - k),
        prior_alpha=prior_alpha,
        prior_beta=prior_beta,
        k=k,
        n=n,
    )


# ════════════════════════════════════════════════════════════════════════
# MLE + CRAMER-RAO
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class NormalMLE:
    mean: float
    mle_std: float  # ddof=0, biased
    unbiased_std: float  # ddof=1, Bessel's correction
    n: int

    @property
    def fisher_information(self) -> float:
        return self.n / self.mle_std**2

    @property
    def cramer_rao_bound(self) -> float:
        return 1.0 / self.fisher_information

    @property
    def standard_error(self) -> float:
        return float(np.sqrt(self.cramer_rao_bound))


def normal_mle(data: np.ndarray) -> NormalMLE:
    """MLE for Normal (μ, σ²) with Cramér-Rao bound bookkeeping."""
    arr = np.asarray(data, dtype=np.float64)
    return NormalMLE(
        mean=float(arr.mean()),
        mle_std=float(arr.std(ddof=0)),
        unbiased_std=float(arr.std(ddof=1)),
        n=len(arr),
    )


# ════════════════════════════════════════════════════════════════════════
# BOOTSTRAP INTERVALS
# ════════════════════════════════════════════════════════════════════════


def bootstrap_mean_distribution(
    data: np.ndarray, n_bootstrap: int = 10_000, seed: int = 42
) -> np.ndarray:
    """Non-parametric bootstrap of the sample mean."""
    rng = np.random.default_rng(seed)
    n = len(data)
    return np.array(
        [rng.choice(data, size=n, replace=True).mean() for _ in range(n_bootstrap)]
    )


def percentile_ci(draws: np.ndarray, level: float = 0.95) -> tuple[float, float]:
    lo = (1 - level) / 2 * 100
    hi = (1 + level) / 2 * 100
    return float(np.percentile(draws, lo)), float(np.percentile(draws, hi))


def bca_ci(
    data: np.ndarray, n_bootstrap: int = 10_000, seed: int = 42, level: float = 0.95
) -> tuple[float, float]:
    """Bias-corrected accelerated bootstrap CI for the mean (via scipy)."""
    result = stats.bootstrap(
        (np.asarray(data, dtype=np.float64),),
        statistic=np.mean,
        n_resamples=n_bootstrap,
        confidence_level=level,
        method="BCa",
        random_state=seed,
    )
    return float(result.confidence_interval.low), float(result.confidence_interval.high)


# ════════════════════════════════════════════════════════════════════════
# FORMATTING
# ════════════════════════════════════════════════════════════════════════


def fmt_money(x: float) -> str:
    return f"${x:,.0f}"


def print_interval(label: str, lo: float, hi: float) -> None:
    print(
        f"  {label:<28} [{fmt_money(lo)}, {fmt_money(hi)}]  width={fmt_money(hi - lo)}"
    )


# ── shared/mlfp02/ex_5.py ──
"""
Shared infrastructure for MLFP02 Exercise 5 — Linear Regression.

Contains: HDB resale data loading, feature engineering, OLS fitting utilities,
diagnostic helpers, and visualisation helpers. Technique-specific code (the
derivation walk-throughs, the WLS weight construction, the polynomial/dummy
matrices) lives in the per-technique files, not here.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ════════════════════════════════════════════════════════════════════════

NUMERIC_FEATURES: list[str] = [
    "floor_area_sqm",
    "storey_midpoint",
    "remaining_lease_years",
]
TARGET: str = "resale_price"
BASE_FLAT_TYPE: str = "3 ROOM"

OUTPUT_DIR = Path("outputs") / "ex5_linear_regression"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale flat transactions
# ════════════════════════════════════════════════════════════════════════


def load_hdb_clean() -> pl.DataFrame:
    """Load HDB resale data, engineer numeric features, drop nulls.

    Returns a polars DataFrame with columns:
      - floor_area_sqm (Float)
      - storey_midpoint (Float, midpoint of '07 TO 09' range)
      - remaining_lease_years (Float, 99 - years_elapsed_since_commence)
      - resale_price (target, SGD)
      - flat_type (categorical)
      - town (categorical)
    """
    loader = MLFPDataLoader()
    hdb = loader.load("mlfp01", "hdb_resale.parquet")

    hdb = hdb.with_columns(
        pl.col("month").str.to_date("%Y-%m").alias("transaction_date"),
    )
    hdb_recent = hdb.filter(pl.col("transaction_date") >= pl.date(2020, 1, 1))

    hdb_recent = hdb_recent.with_columns(
        (
            (
                pl.col("storey_range").str.extract(r"(\d+)", 1).cast(pl.Float64)
                + pl.col("storey_range").str.extract(r"TO (\d+)", 1).cast(pl.Float64)
            )
            / 2
        ).alias("storey_midpoint"),
        (99 - (pl.col("transaction_date").dt.year() - pl.col("lease_commence_date")))
        .cast(pl.Float64)
        .alias("remaining_lease_years"),
    )

    return hdb_recent.drop_nulls(
        subset=[*NUMERIC_FEATURES, TARGET],
    )


def build_design_matrix(
    df: pl.DataFrame, features: list[str] | None = None
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Build (X_with_intercept, y, feature_names) from a cleaned HDB frame."""
    features = features or NUMERIC_FEATURES
    y = df[TARGET].to_numpy().astype(np.float64)
    X_raw = df.select(features).to_numpy().astype(np.float64)
    n_obs = X_raw.shape[0]
    X = np.column_stack([np.ones(n_obs), X_raw])
    feature_names = ["intercept"] + list(features)
    return X, y, feature_names


# ════════════════════════════════════════════════════════════════════════
# OLS CORE — the workhorse every technique reuses
# ════════════════════════════════════════════════════════════════════════


def fit_ols(X: np.ndarray, y: np.ndarray) -> dict[str, Any]:
    """Fit OLS via the normal equation and return core statistics.

    Returns a dict with keys: beta, y_hat, residuals, XtX_inv, SSR, SST, SSE,
    R2, adj_R2, sigma_hat, se_beta, t_stats, p_values, f_stat, f_p_value, n, k.
    """
    n, k = X.shape
    XtX = X.T @ X
    XtX_inv = np.linalg.inv(XtX)
    beta = XtX_inv @ X.T @ y

    y_hat = X @ beta
    residuals = y - y_hat

    SSR = float(np.sum(residuals**2))
    SST = float(np.sum((y - y.mean()) ** 2))
    SSE = float(np.sum((y_hat - y.mean()) ** 2))

    sigma_sq = SSR / (n - k)
    sigma_hat = float(np.sqrt(sigma_sq))
    se_beta = np.sqrt(sigma_sq * np.diag(XtX_inv))
    t_stats = beta / se_beta
    # sf (survival function) keeps precision in the far tail; 1 - cdf
    # rounds to exactly 0 once the tail drops below ~1e-16.
    p_values = 2 * stats.t.sf(np.abs(t_stats), df=n - k)

    r2 = 1 - SSR / SST
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k)
    f_stat = (SSE / (k - 1)) / (SSR / (n - k))
    f_p = stats.f.sf(f_stat, dfn=k - 1, dfd=n - k)

    return {
        "beta": beta,
        "y_hat": y_hat,
        "residuals": residuals,
        "XtX_inv": XtX_inv,
        "SSR": SSR,
        "SST": SST,
        "SSE": SSE,
        "R2": float(r2),
        "adj_R2": float(adj_r2),
        "sigma_hat": sigma_hat,
        "se_beta": se_beta,
        "t_stats": t_stats,
        "p_values": p_values,
        "f_stat": float(f_stat),
        "f_p_value": float(f_p),
        "n": int(n),
        "k": int(k),
    }


def format_p_value(p: float) -> str:
    """Render a p-value for printing, including the underflow case.

    Returns e.g. "= 3.20e-05", or "< 1e-300" when the tail probability is
    smaller than the smallest representable double and evaluates to 0.
    """
    return f"= {p:.2e}" if p > 0 else "< 1e-300"


# ════════════════════════════════════════════════════════════════════════
# K-FOLD CROSS-VALIDATION — fold assignment, from scratch
# ════════════════════════════════════════════════════════════════════════


def kfold_indices(
    n: int, k: int = 5, seed: int = 42
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Return k (train_idx, test_idx) pairs from a seeded shuffle.

    Fold sizes differ by at most one row. The shuffle makes folds
    exchangeable — required because the HDB frame is sorted by month and
    an unshuffled split would make the last fold a different time period
    (that is a DIFFERENT technique: out-of-time validation, ex_8).
    """
    rng = np.random.default_rng(seed)
    order = rng.permutation(n)
    folds = np.array_split(order, k)
    pairs: list[tuple[np.ndarray, np.ndarray]] = []
    for i in range(k):
        test_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        pairs.append((train_idx, test_idx))
    return pairs


def ols_r2_on(
    X: np.ndarray, y: np.ndarray, train_idx: np.ndarray, test_idx: np.ndarray
) -> float:
    """Fit OLS on train_idx, return R² computed on test_idx (out-of-sample)."""
    beta = np.linalg.lstsq(X[train_idx], y[train_idx], rcond=None)[0]
    y_true = y[test_idx]
    y_pred = X[test_idx] @ beta
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    return 1.0 - ss_res / ss_tot


# ════════════════════════════════════════════════════════════════════════
# GEO FEATURES — town centroids and haversine distance to the CBD
# ════════════════════════════════════════════════════════════════════════
#
# The resale file has no coordinates, so locations come from a lookup of
# approximate TOWN-CENTRE coordinates (public geographic facts; teaching
# proxy — a production pipeline would geocode block + street via OneMap).
# Distance is to Raffles Place, the centre of the Central Business
# District.

CBD_RAFFLES_PLACE: tuple[float, float] = (1.2844, 103.8510)

TOWN_CENTROIDS: dict[str, tuple[float, float]] = {
    "ANG MO KIO": (1.3691, 103.8454),
    "BEDOK": (1.3236, 103.9273),
    "BISHAN": (1.3526, 103.8352),
    "BOON LAY": (1.3366, 103.7039),
    "BUKIT BATOK": (1.3490, 103.7496),
    "BUKIT MERAH": (1.2819, 103.8239),
    "BUKIT PANJANG": (1.3774, 103.7719),
    "BUKIT TIMAH": (1.3294, 103.8021),
    "CENTRAL AREA": (1.2903, 103.8520),
    "CHOA CHU KANG": (1.3840, 103.7470),
    "CLEMENTI": (1.3162, 103.7649),
    "GEYLANG": (1.3201, 103.8871),
    "HOUGANG": (1.3612, 103.8863),
    "JURONG EAST": (1.3329, 103.7436),
    "JURONG WEST": (1.3404, 103.7090),
    "KALLANG/WHAMPOA": (1.3100, 103.8651),
    "MARINE PARADE": (1.3017, 103.9057),
    "PASIR RIS": (1.3721, 103.9474),
    "PUNGGOL": (1.3984, 103.9072),
    "QUEENSTOWN": (1.2942, 103.7861),
    "SEMBAWANG": (1.4491, 103.8185),
    "SENGKANG": (1.3868, 103.8914),
    "SERANGOON": (1.3554, 103.8679),
    "TAMPINES": (1.3496, 103.9568),
    "TOA PAYOH": (1.3343, 103.8563),
    "WOODLANDS": (1.4382, 103.7890),
    "YISHUN": (1.4304, 103.8354),
}


def haversine_km(
    lat1: np.ndarray, lon1: np.ndarray, lat2: float, lon2: float
) -> np.ndarray:
    """Great-circle distance in km (mean Earth radius 6371 km)."""
    r = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2.0 * r * np.arcsin(np.sqrt(a))


def add_geo_features(df: pl.DataFrame) -> pl.DataFrame:
    """Add town centroid lat/lon and haversine distance-to-CBD (km).

    Rows whose town is missing from the centroid lookup become null and
    are dropped by the model frame's drop_nulls downstream.
    """
    towns = pl.DataFrame(
        {
            "town": list(TOWN_CENTROIDS.keys()),
            "town_lat": [c[0] for c in TOWN_CENTROIDS.values()],
            "town_lon": [c[1] for c in TOWN_CENTROIDS.values()],
        }
    )
    out = df.join(towns, on="town", how="left")
    lat = out["town_lat"].to_numpy().astype(np.float64)
    lon = out["town_lon"].to_numpy().astype(np.float64)
    dist = haversine_km(lat, lon, *CBD_RAFFLES_PLACE)
    return out.with_columns(
        pl.Series("dist_to_cbd_km", dist).cast(pl.Float64)
    )


# ════════════════════════════════════════════════════════════════════════
# EXPERIMENT TRACKING — kailash-ml ExperimentTracker
# ════════════════════════════════════════════════════════════════════════

TRACKER_STORE_URL = (
    f"sqlite:///{(OUTPUT_DIR / 'experiments.db').resolve().as_posix()}"
)


def track_train_run(
    experiment: str,
    run_name: str,
    params: dict[str, str],
    metrics: dict[str, float],
) -> str:
    """Log one Train-phase run to ExperimentTracker (sync wrapper).

    Returns the run_id. The tracker is closed in a finally block — kailash-ml
    holds the store connection open until close() is called.
    """
    import asyncio

    async def _log() -> str:
        from kailash_ml import ExperimentTracker

        tracker = await ExperimentTracker.create(store_url=TRACKER_STORE_URL)
        try:
            async with tracker.track(
                experiment=experiment, run_name=run_name
            ) as run:
                await run.log_params(params)
                await run.log_metrics(metrics)
                return run.run_id
        finally:
            await tracker.close()

    return asyncio.run(_log())


def print_coef_table(names: list[str], fit: dict[str, Any]) -> None:
    """Print coefficient / SE / t / p table for an OLS fit."""
    beta = fit["beta"]
    se = fit["se_beta"]
    t = fit["t_stats"]
    p = fit["p_values"]
    print(f"\n{'Feature':<25} {'β':>14} {'SE(β)':>12} {'t':>8} {'p':>10} {'Sig':>4}")
    print("-" * 78)
    for i, name in enumerate(names):
        if p[i] < 0.001:
            sig = "***"
        elif p[i] < 0.01:
            sig = "**"
        elif p[i] < 0.05:
            sig = "*"
        else:
            sig = "ns"
        print(
            f"{name:<25} {beta[i]:>14,.2f} {se[i]:>12,.2f} "
            f"{t[i]:>8.2f} {(f'{p[i]:.2e}' if p[i] > 0 else '<1e-300'):>10} {sig:>4}"
        )


# ════════════════════════════════════════════════════════════════════════
# DIAGNOSTICS — VIF, Breusch-Pagan, residual shape
# ════════════════════════════════════════════════════════════════════════


def compute_vif(X_raw: np.ndarray, feature_names: list[str]) -> dict[str, float]:
    """Variance Inflation Factor for each feature (no intercept column)."""
    n = X_raw.shape[0]
    results: dict[str, float] = {}
    for j in range(X_raw.shape[1]):
        other = [i for i in range(X_raw.shape[1]) if i != j]
        Xo = np.column_stack([np.ones(n), X_raw[:, other]])
        yj = X_raw[:, j]
        beta_j = np.linalg.lstsq(Xo, yj, rcond=None)[0]
        yhat_j = Xo @ beta_j
        ss_res = np.sum((yj - yhat_j) ** 2)
        ss_tot = np.sum((yj - yj.mean()) ** 2)
        r2_j = 1 - ss_res / ss_tot
        results[feature_names[j]] = (
            float(1.0 / (1.0 - r2_j)) if r2_j < 1 else float("inf")
        )
    return results


def breusch_pagan(residuals: np.ndarray, X_raw: np.ndarray) -> tuple[float, float]:
    """Breusch-Pagan test for heteroscedasticity. Returns (BP statistic, p-value)."""
    n = X_raw.shape[0]
    e_sq = residuals**2
    Xbp = np.column_stack([np.ones(n), X_raw])
    beta = np.linalg.lstsq(Xbp, e_sq, rcond=None)[0]
    pred = Xbp @ beta
    sse = np.sum((e_sq - pred) ** 2)
    sst = np.sum((e_sq - e_sq.mean()) ** 2)
    r2 = 1 - sse / sst
    bp_stat = n * r2
    p = stats.chi2.sf(bp_stat, df=X_raw.shape[1])
    return float(bp_stat), float(p)


# ════════════════════════════════════════════════════════════════════════
# VISUALISE — residual plots + actual-vs-predicted
# ════════════════════════════════════════════════════════════════════════


def save_residual_diagnostics(
    y_hat: np.ndarray,
    residuals: np.ndarray,
    feature_col: np.ndarray,
    feature_label: str,
    filename: str,
) -> Path:
    """Four-panel diagnostic figure: residuals vs fitted, histogram, Q-Q, vs feature."""
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[
            "Residuals vs Fitted",
            "Residual Histogram",
            "Q-Q Plot",
            f"Residuals vs {feature_label}",
        ],
    )
    sample = min(3000, len(residuals))
    fig.add_trace(
        go.Scatter(
            x=y_hat[:sample],
            y=residuals[:sample],
            mode="markers",
            marker={"size": 2, "opacity": 0.3},
            name="Residuals",
        ),
        row=1,
        col=1,
    )
    fig.add_hline(y=0, row=1, col=1, line_dash="dash")
    fig.add_trace(go.Histogram(x=residuals, nbinsx=50, name="Residuals"), row=1, col=2)
    sorted_resid = np.sort(residuals)
    step = max(1, len(sorted_resid) // 2000)
    theoretical = stats.norm.ppf(np.linspace(0.001, 0.999, len(sorted_resid)))
    fig.add_trace(
        go.Scatter(
            x=theoretical[::step],
            y=sorted_resid[::step],
            mode="markers",
            marker={"size": 2},
            name="Q-Q",
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=feature_col[:sample],
            y=residuals[:sample],
            mode="markers",
            marker={"size": 2, "opacity": 0.3},
            name=f"vs {feature_label}",
        ),
        row=2,
        col=2,
    )
    fig.update_layout(height=600, title="Residual Diagnostics", showlegend=False)
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


def save_actual_vs_predicted(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    title: str,
    filename: str,
) -> Path:
    """Actual-vs-predicted scatter with the perfect-prediction diagonal."""
    sample = min(2000, len(y_true))
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=y_true[:sample],
            y=y_pred[:sample],
            mode="markers",
            marker={"size": 3, "opacity": 0.4},
            name="Predictions",
        )
    )
    lo, hi = float(y_true.min()), float(y_true.max())
    fig.add_trace(
        go.Scatter(
            x=[lo, hi],
            y=[lo, hi],
            mode="lines",
            line={"dash": "dash", "color": "red"},
            name="Perfect",
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="Actual Price (SGD)",
        yaxis_title="Predicted Price (SGD)",
        height=500,
    )
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


# ── shared/mlfp02/ex_6.py ──
"""Shared infrastructure for MLFP02 Exercise 6 — Logistic Regression and
Classification Foundations.

Contains: HDB data loading, binary target construction, design matrix
preparation, sigmoid implementation, the neg-log-likelihood + gradient used
by scipy.optimize, calibration-curve binning, and output-directory setup.

Technique-specific code (odds ratios, threshold sweeps, confusion matrices,
ROC/PR/calibration plots, ANOVA + Tukey HSD) does NOT belong here — it
lives in the per-technique files in solutions/ex_6/ and local/ex_6/.
"""

from pathlib import Path

import numpy as np
import polars as pl


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "mlfp02_ex6_logistic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS: list[str] = ["floor_area_sqm", "storey_mid", "remaining_lease"]


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale transactions (Singapore)
# ════════════════════════════════════════════════════════════════════════


def load_hdb_recent() -> pl.DataFrame:
    """Load HDB resale transactions filtered to 2020+.

    Returns a polars DataFrame with the raw columns from the dataset
    (resale_price, flat_type, floor_area_sqm, storey_range, lease_commence_date,
    month, etc.). No target construction — see build_classification_frame().
    """
    loader = MLFPDataLoader()
    hdb = loader.load("mlfp01", "hdb_resale.parquet")
    return hdb.filter(pl.col("month").str.to_date("%Y-%m") >= pl.date(2020, 1, 1))


def build_classification_frame(hdb_recent: pl.DataFrame) -> tuple[pl.DataFrame, float]:
    """Add the binary target and engineered features used by logistic regression.

    The target ``high_price`` = 1 if resale_price > median, 0 otherwise — this
    gives a balanced ~50/50 split so the baseline accuracy is 50%.

    Returns (dataframe, median_price).
    """
    median_price = hdb_recent["resale_price"].median()
    frame = hdb_recent.with_columns(
        (pl.col("resale_price") > median_price).cast(pl.Int32).alias("high_price"),
        (
            (
                pl.col("storey_range").str.extract(r"(\d+)", 1).cast(pl.Float64)
                + pl.col("storey_range").str.extract(r"TO (\d+)", 1).cast(pl.Float64)
            )
            / 2.0
        ).alias("storey_mid"),
        (
            99
            - (
                pl.col("month").str.to_date("%Y-%m").dt.year()
                - pl.col("lease_commence_date")
            )
        )
        .cast(pl.Float64)
        .alias("remaining_lease"),
    ).drop_nulls(
        subset=["floor_area_sqm", "storey_mid", "high_price", "remaining_lease"]
    )
    return frame, float(median_price)


def build_design_matrix(
    frame: pl.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str]]:
    """Standardise features and append an intercept column.

    Returns (X_with_intercept, y, X_mean, X_std, feature_names_with_intercept).
    """
    X_raw = frame.select(FEATURE_COLS).to_numpy().astype(np.float64)
    y = frame["high_price"].to_numpy().astype(np.float64)

    X_mean = X_raw.mean(axis=0)
    X_std = X_raw.std(axis=0)
    X_scaled = (X_raw - X_mean) / X_std

    n_obs = X_scaled.shape[0]
    X = np.column_stack([np.ones(n_obs), X_scaled])
    feature_names = ["intercept", *FEATURE_COLS]
    return X, y, X_mean, X_std, feature_names


# ════════════════════════════════════════════════════════════════════════
# SIGMOID + BERNOULLI LOG-LIKELIHOOD
# ════════════════════════════════════════════════════════════════════════


def sigmoid(z: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid σ(z) = 1 / (1 + exp(-z)).

    For z ≥ 0 use 1/(1+exp(-z)); for z < 0 use exp(z)/(1+exp(z)). Both
    branches avoid overflow at the extremes.
    """
    z = np.asarray(z, dtype=np.float64)
    result = np.zeros_like(z)
    pos = z >= 0
    neg = ~pos
    result[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    exp_z = np.exp(z[neg])
    result[neg] = exp_z / (1.0 + exp_z)
    return result


def neg_log_likelihood_logistic(
    beta: np.ndarray, X: np.ndarray, y: np.ndarray
) -> float:
    """Negative Bernoulli log-likelihood: -Σ[y log p + (1-y) log(1-p)]."""
    z = X @ beta
    p = sigmoid(z)
    p = np.clip(p, 1e-15, 1 - 1e-15)
    ll = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return float(-ll)


def neg_ll_gradient(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Gradient of the negative log-likelihood: -X^T (y - p)."""
    z = X @ beta
    p = sigmoid(z)
    return -X.T @ (y - p)


# ════════════════════════════════════════════════════════════════════════
# ORIGINAL-SCALE COEFFICIENT CONVERSION
# ════════════════════════════════════════════════════════════════════════


def unscale_coefficients(
    beta_scaled: np.ndarray, X_mean: np.ndarray, X_std: np.ndarray
) -> np.ndarray:
    """Convert coefficients fit on standardised features to the original scale.

    β_original[j]   = β_scaled[j] / σ_j  for j ≥ 1
    β_original[0]   = β_scaled[0] - Σ β_scaled[j] * μ_j / σ_j
    """
    beta_original = np.zeros_like(beta_scaled)
    beta_original[0] = beta_scaled[0] - float(np.sum(beta_scaled[1:] * X_mean / X_std))
    beta_original[1:] = beta_scaled[1:] / X_std
    return beta_original


# ════════════════════════════════════════════════════════════════════════
# CALIBRATION BINNING
# ════════════════════════════════════════════════════════════════════════


def calibration_bins(
    y: np.ndarray, p: np.ndarray, n_bins: int = 10
) -> tuple[list[float], list[float], list[int]]:
    """Equal-width binning over predicted probabilities.

    Returns (mean_predicted, mean_observed, counts) for non-empty bins only.
    """
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    mean_pred: list[float] = []
    mean_obs: list[float] = []
    counts: list[int] = []
    for i in range(n_bins):
        hi = edges[i + 1] + (1e-12 if i == n_bins - 1 else 0.0)
        mask = (p >= edges[i]) & (p < hi)
        if mask.sum() > 0:
            mean_pred.append(float(p[mask].mean()))
            mean_obs.append(float(y[mask].mean()))
            counts.append(int(mask.sum()))
    return mean_pred, mean_obs, counts


# ════════════════════════════════════════════════════════════════════════
# MULTINOMIAL LOGIT — softmax generalisation of the binary model
# ════════════════════════════════════════════════════════════════════════


def softmax_rows(Z: np.ndarray) -> np.ndarray:
    """Row-wise softmax with the max-subtraction stabiliser.

    Z has shape (n, K); returns P of the same shape with rows summing to 1.
    """
    Zs = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Zs)
    return e / e.sum(axis=1, keepdims=True)


def neg_log_likelihood_multinomial(
    beta_flat: np.ndarray, X: np.ndarray, y_idx: np.ndarray, K: int
) -> float:
    """Negative multinomial log-likelihood with the reference-class
    parameterisation: class 0's coefficient vector is fixed at zero, so
    beta_flat holds (K-1) * n_features free parameters."""
    n_feat = X.shape[1]
    B = np.zeros((n_feat, K))
    B[:, 1:] = beta_flat.reshape(n_feat, K - 1)
    P = softmax_rows(X @ B)
    ll = float(np.log(np.clip(P[np.arange(len(y_idx)), y_idx], 1e-15, 1.0)).sum())
    return -ll


def multinomial_gradient(
    beta_flat: np.ndarray, X: np.ndarray, y_idx: np.ndarray, K: int
) -> np.ndarray:
    """Gradient of the multinomial NLL w.r.t. the free (non-reference)
    coefficient block, flattened to match beta_flat."""
    n_feat = X.shape[1]
    B = np.zeros((n_feat, K))
    B[:, 1:] = beta_flat.reshape(n_feat, K - 1)
    P = softmax_rows(X @ B)
    Y = np.zeros_like(P)
    Y[np.arange(len(y_idx)), y_idx] = 1.0
    grad_full = -X.T @ (Y - P)  # (n_feat, K); column 0 is the fixed class
    return grad_full[:, 1:].ravel()


FLAT_TYPE_CLASSES: list[str] = ["3 ROOM", "4 ROOM", "5 ROOM"]
MULTINOMIAL_FEATURES: list[str] = [
    "resale_price",
    "storey_mid",
    "remaining_lease",
    "dist_to_cbd_km",
]


def load_flat_type_frame() -> pl.DataFrame:
    """HDB 2020+ 3/4/5-room transactions with context features (NO floor
    area — flat type is defined by area, so including it would be
    tautological; the task is inferring type from market context)."""

    hdb = load_hdb_all()
    hdb = hdb.filter(
        (pl.col("resale_price") >= 100_000) & (pl.col("resale_price") <= 5_000_000)
    )
    frame = hdb.filter(pl.col("flat_type").is_in(FLAT_TYPE_CLASSES))
    frame = frame.with_columns(
        (
            (
                pl.col("storey_range").str.extract(r"(\d+)", 1).cast(pl.Float64)
                + pl.col("storey_range").str.extract(r"TO (\d+)", 1).cast(pl.Float64)
            )
            / 2.0
        ).alias("storey_mid"),
        (
            99
            - (
                pl.col("month").str.to_date("%Y-%m").dt.year()
                - pl.col("lease_commence_date")
            )
        )
        .cast(pl.Float64)
        .alias("remaining_lease"),
    )
    frame = add_geo_features(frame)
    return frame.drop_nulls(subset=[*MULTINOMIAL_FEATURES, "flat_type"])


# ════════════════════════════════════════════════════════════════════════
# EXPERIMENT TRACKING — kailash-ml ExperimentTracker
# ════════════════════════════════════════════════════════════════════════

TRACKER_STORE_URL = (
    f"sqlite:///{(OUTPUT_DIR / 'experiments.db').resolve().as_posix()}"
)


def track_train_run(
    experiment: str,
    run_name: str,
    params: dict[str, str],
    metrics: dict[str, float],
) -> str:
    """Log one Train-phase run to ExperimentTracker (sync wrapper).

    Returns the run_id. The tracker is closed in a finally block — kailash-ml
    holds the store connection open until close() is called.
    """
    import asyncio

    async def _log() -> str:
        from kailash_ml import ExperimentTracker

        tracker = await ExperimentTracker.create(store_url=TRACKER_STORE_URL)
        try:
            async with tracker.track(
                experiment=experiment, run_name=run_name
            ) as run:
                await run.log_params(params)
                await run.log_metrics(metrics)
                return run.run_id
        finally:
            await tracker.close()

    return asyncio.run(_log())




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 6.5: Multinomial Logistic Regression
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Generalise binary logistic to K classes: softmax + reference class
#   - Why one class is the reference (identification: only K-1 logit
#     vectors are free — the Kth is fixed at zero)
#   - Fit multinomial MLE from scratch (NLL + analytical gradient, L-BFGS)
#   - Read odds ratios against the reference class: exp(β) per feature
#   - Compare against sklearn's multinomial oracle and a one-vs-rest
#     baseline (the framing alternative from the lesson)
#
# PREREQUISITES: 01_logistic_from_scratch.py (sigmoid, Bernoulli NLL)
#
# ESTIMATED TIME: ~45 min
#
# TASKS:
#   1. Theory — from two classes to K: softmax and the reference class
#   2. Build — 3/4/5-room transactions with context features (no area!)
#   3. Train — multinomial MLE from scratch + sklearn oracle agreement
#   4. Visualise — class-probability profiles across price
#   5. Apply — an agent infers flat type when the listing omits area
#
# ─── FRAMEWORK-FIRST EXEMPTION ──────────────────────────────────────────
# As in 01_logistic_from_scratch.py, sklearn appears ONCE as a
# correctness oracle for the from-scratch optimiser. M2 teaches logistic
# models as inference; production engines arrive in M3.
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression  # exemption: oracle only
from sklearn.metrics import accuracy_score  # exemption: stateless utility



## THEORY — From Two Classes to K: Softmax and the Reference Class


In [ ]:
# Binary logistic models ONE logit: log(P₁/P₀) = xβ. With K classes we
# need K-1 logits against a REFERENCE class (class 0):
#
#   log(P_k / P_0) = xβ_k     for k = 1, …, K-1
#   P_k = exp(xβ_k) / (1 + Σ_j exp(xβ_j))        ← the softmax
#
# The reference class is not optional: adding a constant vector c to ALL
# β_k leaves every probability unchanged, so the model is unidentified
# until one class is pinned at β₀ = 0. Software does this silently;
# doing it by hand makes the odds-ratio reading possible:
#
#   exp(β_kj) = how the ODDS of class k vs the reference multiply for
#               each unit of feature j, holding the others constant
#
# Why no floor_area_sqm in the features? Flat type is DEFINED by floor
# area band — an area feature would score ~100% and teach nothing about
# estimation. The honest task: infer the type from MARKET CONTEXT
# (price, storey, lease, distance) — the situation an agent faces when a
# listing omits the floor size.



## TASK 2 — BUILD: The flat-type frame


In [ ]:
print("\n" + "=" * 70)
print("  MLFP02 Exercise 6.5: Multinomial Logistic Regression")
print("=" * 70)

frame = load_flat_type_frame()
K = len(FLAT_TYPE_CLASSES)
print(f"\n  Rows: {frame.height:,} (2020+, sentinel prices removed)")
for cls in FLAT_TYPE_CLASSES:
    share = frame.filter(frame["flat_type"] == cls).height / frame.height
    print(f"    {cls}: {share:.1%}")
print(f"  Features (market context, no floor area): {MULTINOMIAL_FEATURES}")
print(f"  Reference class: {FLAT_TYPE_CLASSES[0]} (β fixed at 0)")

X_raw = frame.select(MULTINOMIAL_FEATURES).to_numpy().astype(np.float64)
y_idx = np.array(
    [FLAT_TYPE_CLASSES.index(v) for v in frame["flat_type"].to_list()]
)

# Standardise (same discipline as 6.1) and append the intercept
X_mean = X_raw.mean(axis=0)
X_std = X_raw.std(axis=0)
X = np.column_stack([np.ones(len(y_idx)), (X_raw - X_mean) / X_std])
n_feat = X.shape[1]



### Checkpoint 1


In [ ]:
assert X.shape == (len(y_idx), len(MULTINOMIAL_FEATURES) + 1), "Design shape"
assert set(np.unique(y_idx).tolist()) == {0, 1, 2}, "All three classes present"
assert np.isfinite(X).all(), "Design matrix must be finite"
print("\n✓ Checkpoint 1 passed — multinomial frame built\n")



## TASK 3 — TRAIN: Multinomial MLE from scratch + oracle agreement


In [ ]:
beta0 = np.zeros(n_feat * (K - 1))
result = minimize(
    neg_log_likelihood_multinomial,
    beta0,
    args=(X, y_idx, K),
    jac=multinomial_gradient,
    method="L-BFGS-B",
    options={"maxiter": 500},
)
assert result.success, f"Multinomial optimisation failed: {result.message}"

B_hat = np.zeros((n_feat, K))
B_hat[:, 1:] = result.x.reshape(n_feat, K - 1)
P_hat = softmax_rows(X @ B_hat)
y_pred = P_hat.argmax(axis=1)
acc_scratch = float((y_pred == y_idx).mean())
base_rate = float(np.bincount(y_idx).max() / len(y_idx))

print("=== Multinomial MLE (from scratch) ===")
print(f"  converged: {result.success}, final NLL: {result.fun:,.1f}")
print(f"  in-sample accuracy: {acc_scratch:.3f}  (base rate: {base_rate:.3f})")

# sklearn oracle, matched to the from-scratch model: same standardised design
# and NO regularisation (sklearn defaults to L2, which would bias the
# comparison). multinomial is the default for lbfgs.
X_std_design = (X_raw - X_mean) / X_std
oracle = LogisticRegression(max_iter=2000, C=1e12, fit_intercept=True)
oracle.fit(X_std_design, y_idx)
acc_oracle = float(accuracy_score(y_idx, oracle.predict(X_std_design)))
print(f"  sklearn oracle accuracy: {acc_oracle:.3f}")

# Agreement on predicted class labels (evaluate on the same design the
# oracle was fitted on — the standardised one, not raw).
P_oracle = oracle.predict_proba(X_std_design)  # columns follow oracle.classes_
oracle_classes = list(oracle.classes_)
P_oracle_aligned = np.column_stack(
    [P_oracle[:, oracle_classes.index(k)] for k in range(K)]
)
pred_oracle = P_oracle_aligned.argmax(axis=1)
label_agreement = float((pred_oracle == y_pred).mean())
print(f"  label agreement with oracle: {label_agreement:.1%}")

# Odds ratios vs the reference class, per feature (standardised scale)
print(f"\n=== Odds ratios vs {FLAT_TYPE_CLASSES[0]} (per SD of feature) ===")
print(f"{'Feature':<18} {'OR 4-room':>10} {'OR 5-room':>10}")
print("-" * 42)
for j, feat in enumerate(["intercept", *MULTINOMIAL_FEATURES]):
    if feat == "intercept":
        continue
    or_4 = float(np.exp(B_hat[j, 1]))
    or_5 = float(np.exp(B_hat[j, 2]))
    print(f"{feat:<18} {or_4:>10.3f} {or_5:>10.3f}")
print(
    "\n  OR > 1: one SD more of the feature multiplies the odds of that\n"
    "  class vs 3-room by that factor. OR < 1: the odds shrink. Price\n"
    "  and distance carry the type signal when area is hidden."
)

# One-vs-rest comparison (the lesson's framing alternative): three
# separate binary logits, then normalise their probabilities
ovr_probs = np.zeros_like(P_hat)
for k in range(K):

    y_bin = (y_idx == k).astype(np.float64)
    res_k = minimize(
        neg_log_likelihood_logistic,
        np.zeros(n_feat),
        args=(X, y_bin),
        jac=neg_ll_gradient,
        method="L-BFGS-B",
    )
    z_k = X @ res_k.x

    ovr_probs[:, k] = sigmoid(z_k)
ovr_probs = ovr_probs / ovr_probs.sum(axis=1, keepdims=True)
acc_ovr = float((ovr_probs.argmax(axis=1) == y_idx).mean())
print(f"\n  One-vs-rest accuracy: {acc_ovr:.3f}")
print(
    "  One-vs-rest fits K independent binaries and renormalises — simple,\n"
    "  but the binaries are not jointly calibrated. Softmax shares the\n"
    "  denominator across classes, which is why it is the default."
)



### Log the fit to ExperimentTracker


In [ ]:
run_id = track_train_run(
    experiment="mlfp02_ex6_05_multinomial",
    run_name="flat_type_context_softmax",
    params={
        "classes": ",".join(FLAT_TYPE_CLASSES),
        "features": ",".join(MULTINOMIAL_FEATURES),
        "reference_class": FLAT_TYPE_CLASSES[0],
        "optimiser": "L-BFGS-B analytical gradient",
    },
    metrics={
        "nll_final": float(result.fun),
        "accuracy_scratch": acc_scratch,
        "accuracy_oracle": acc_oracle,
        "accuracy_ovr": acc_ovr,
        "label_agreement_oracle": label_agreement,
        "base_rate": base_rate,
    },
)
print(f"\nLogged training run to ExperimentTracker (run {run_id})")



### Checkpoint 2


In [ ]:
assert acc_scratch > base_rate, "Model must beat the majority-class base rate"
assert abs(acc_scratch - acc_oracle) < 0.02, (
    "From-scratch and oracle accuracies must agree within 2 points"
)
assert label_agreement > 0.95, "Label agreement with oracle must exceed 95%"
assert np.allclose(P_hat.sum(axis=1), 1.0), "Softmax rows must sum to 1"
print("\n✓ Checkpoint 2 passed — multinomial fit agrees with the oracle\n")



## TASK 4 — VISUALISE: Class-probability profiles across price


In [ ]:
# Sweep resale_price across its observed range, other features at medians
price_grid_raw = np.linspace(X_raw[:, 0].min(), np.percentile(X_raw[:, 0], 99), 200)
medians_raw = np.median(X_raw, axis=0)
profiles = np.tile(medians_raw, (len(price_grid_raw), 1))
profiles[:, 0] = price_grid_raw
profiles_std = (profiles - X_mean) / X_std
P_prof = softmax_rows(np.column_stack([np.ones(len(profiles)), profiles_std]) @ B_hat)

fig = go.Figure()
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for k, cls in enumerate(FLAT_TYPE_CLASSES):
    fig.add_trace(
        go.Scatter(
            x=price_grid_raw,
            y=P_prof[:, k],
            mode="lines",
            name=f"P({cls})",
            line={"color": colors[k]},
        )
    )
fig.update_layout(
    title=(
        "Class Probabilities vs Resale Price "
        "(storey, lease, distance held at medians)"
    ),
    xaxis_title="Resale price ($)",
    yaxis_title="Predicted probability",
    height=450,
)
fig_path = OUTPUT_DIR / "multinomial_profiles.html"
fig.write_html(str(fig_path))
print(f"Saved: {fig_path}")



### Checkpoint 3


In [ ]:
assert fig_path.exists(), "Figure must be written"
assert np.allclose(P_prof.sum(axis=1), 1.0), "Profile rows must sum to 1"
print("\n✓ Checkpoint 3 passed — visualisation saved\n")



## TASK 5 — APPLY: Inferring Flat Type When the Listing Omits Area


In [ ]:
# A property portal (anonymised) ingests listings from partners whose
# feeds sometimes drop the floor-area field. Before the listing goes
# live, the portal wants a type suggestion with calibrated confidence —
# and a rule for when to suppress the suggestion entirely.

print("=== APPLICATION: Listing with missing floor area ===")
new_listing_raw = np.array([[620_000.0, 10.0, 75.0, 12.5]])  # price, storey, lease, km
new_std = (new_listing_raw - X_mean) / X_std
p_new = softmax_rows(np.column_stack([[1.0], new_std]) @ B_hat)[0]
print(
    f"  Listing: $620K, storey ~10, lease 75y, 12.5 km from CBD\n"
    f"  Suggestion: "
    + ", ".join(
        f"P({cls}) = {p_new[k]:.2f}" for k, cls in enumerate(FLAT_TYPE_CLASSES)
    )
)
confidence = float(p_new.max())
threshold = 0.60
print(f"\n  Confidence threshold for display: {threshold:.0%}")
if confidence >= threshold:
    print(
        f"  → Suggest '{FLAT_TYPE_CLASSES[int(p_new.argmax())]}' "
        f"(confidence {confidence:.0%})"
    )
else:
    print(
        f"  → Top class only reaches {confidence:.0%} — SUPPRESS the "
        "suggestion and ask the partner to supply the area."
    )

# Calibration honesty: how often is the model's top class right at
# different confidence levels?
conf_all = P_hat.max(axis=1)
correct_all = (P_hat.argmax(axis=1) == y_idx)
print(f"\n  Empirical reliability on 2020+ data:")
for lo, hi in ((0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.01)):
    m = (conf_all >= lo) & (conf_all < hi)
    if m.sum() > 0:
        print(
            f"    confidence {lo:.1f}–{min(hi, 1.0):.1f}: "
            f"accuracy {correct_all[m].mean():.2f} on {m.sum():,} rows"
        )



### Checkpoint 4


In [ ]:
assert 0 <= confidence <= 1, "Confidence must be a probability"
assert abs(float(p_new.sum()) - 1.0) < 1e-9, "Class probabilities must sum to 1"
print("\n✓ Checkpoint 4 passed — application complete\n")



## REFLECTION


✓ Softmax generalises the sigmoid: K-1 free logits against a pinned
    reference class — identification is not optional
  ✓ From-scratch multinomial MLE (NLL + analytical gradient, L-BFGS)
    matches the sklearn oracle within 2 accuracy points and >95% label
    agreement
  ✓ Odds ratios vs the reference class read exactly as in the binary
    case: exp(β) multiplies the odds per unit (here, per SD)
  ✓ One-vs-rest is the simpler framing but its probabilities are not
    jointly calibrated — softmax shares the denominator
  ✓ Withholding floor area turned a tautology into a real inference
    task — and the confidence-threshold rule (suppress below 60%)
    connects probability quality to product behaviour

  NEXT: Exercise 7 returns to experiments — CUPED variance reduction,
  Bayesian A/B decisions, sequential testing, and DiD.


In [ ]:
print("═" * 70)
print("  WHAT YOU'VE MASTERED (6.5 — Multinomial Logistic Regression)")
print("═" * 70)
print(
)

print("\n✓ Exercise 6.5 complete — Multinomial Logistic Regression")

